# Lesson 25: Multi-Modal Networks — CLIP & Zero-Shot Classification

In this notebook, you will:
1. Load HuggingFace **`openai/clip-vit-base-patch32`** on Colab Free Tier T4 GPU.
2. Compute image–text cosine similarities for **Zero-Shot Image Classification** and **Text-Based Image Search**.

In [ ]:
!pip install -q transformers pillow
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# Create a synthetic red-sky sunset image (or load any local PIL image)
arr = np.zeros((224, 224, 3), dtype=np.uint8)
arr[:112, :, 0] = 240  # warm red sky
arr[:112, :, 1] = 100
arr[112:, :, 2] = 180  # blue ocean
image = Image.fromarray(arr)

labels = ["a sunset over the ocean", "a snowy mountain", "a black and white cat"]
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    probs = model(**inputs).logits_per_image.softmax(dim=1)[0]

for label, p in zip(labels, probs.tolist()):
    print(f"{label:28s}: {p*100:.2f}%")